In [2]:
spark.stop()

In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
# sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
# sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
# sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
# sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
# sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
# sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getO

spark session created


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# 2. Create sample "inline" data as a list of strings (simulating a CSV file)
# We have 6 records with various types of 'N/A' and 'missing' data
raw_data = [
    "1, 5000.50",    # Valid
    "2, N/A",        # Should become NULL
    "3, 7500.00",    # Valid
    "4, N/A",        # Should become NULL
    "5, 1200.25",    # Valid
    "6, 0.00"        # Valid
]

rdd_data = sc.parallelize(raw_data)

In [ ]:
df = spark.read.csv(rdd_data,inferSchema=True)
df.printSchema()
df.show()

In [ ]:
df = spark.read.csv(rdd_data,inferSchema=True).toDF("id", "amount")
df.printSchema()
df.show()

In [ ]:

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("amount", DoubleType(), True)
])

df = spark.read.csv(rdd_data, schema=schema)
df.printSchema()
df.show()


In [ ]:
from pyspark.sql import functions as F

print("Aggregation Result (Average Amount):")
df.select(F.sum("amount").alias("average_transaction")).show()


# 2nd example

In [36]:
data_list = [
    "99.99, john smith, ORD-001",
    "N/A, jane doe, ORD-002",
    "null, bob brown, ORD-003",
    "199, alice green, ORD-004",
    "150.75, charlie black, ORD-005"
]
rdd_data = spark.sparkContext.parallelize(data_list)
rdd_data.take(5)

['99.99, john smith, ORD-001',
 'N/A, jane doe, ORD-002',
 'null, bob brown, ORD-003',
 '199, alice green, ORD-004',
 '150.75, charlie black, ORD-005']

In [33]:
spark.read.csv(rdd_data, inferSchema=True).show()

+--------+--------------+--------+
|     _c0|           _c1|     _c2|
+--------+--------------+--------+
|   99.99|    john smith| ORD-001|
|     N/A|      jane doe| ORD-002|
|    null|     bob brown| ORD-003|
|     199|   alice green| ORD-004|
|'150.75'| charlie black| ORD-005|
+--------+--------------+--------+



In [34]:
df1= spark.read.csv(rdd_data, inferSchema=True).toDF("amount", "customer_name", "order_id")
df1.printSchema()
df1.show()

root
 |-- amount: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_id: string (nullable = true)

+--------+--------------+--------+
|  amount| customer_name|order_id|
+--------+--------------+--------+
|   99.99|    john smith| ORD-001|
|     N/A|      jane doe| ORD-002|
|    null|     bob brown| ORD-003|
|     199|   alice green| ORD-004|
|'150.75'| charlie black| ORD-005|
+--------+--------------+--------+



# using different style to define the spark schema

In [ ]:
user_schema = "amount DOUBLE, customer_name STRING, order_id STRING"
df2= spark.read.csv(rdd_data, schema=user_schema)
df2.printSchema()
df2.show()

root
 |-- amount: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_id: string (nullable = true)

+------+--------------+--------+
|amount| customer_name|order_id|
+------+--------------+--------+
| 99.99|    john smith| ORD-001|
|  null|      jane doe| ORD-002|
|  null|     bob brown| ORD-003|
| 199.0|   alice green| ORD-004|
|150.75| charlie black| ORD-005|
+------+--------------+--------+



26/04/29 09:46:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 4390687 ms exceeds timeout 120000 ms
26/04/29 09:46:35 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Futures timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:103)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1053)
	at org.apache.spark.

In [31]:
from pyspark.sql import functions as f
df2.select(f.sum("amount").alias("sum_of_amount")).show()

+-------------+
|sum_of_amount|
+-------------+
|       449.74|
+-------------+

